# Notebook 01 — Synthetic Data Generation

**Purpose:** Generate the synthetic infrastructure log dataset used throughout this project.

**Output:** `data/raw/infra_logs.csv` (~43,200 rows, 15 servers, 30 days at 15-min intervals)

> ⚠️ **Label isolation note:** `is_anomaly` and `anomaly_type` are generated here as ground truth
> for verification purposes only. They are **never** used as model inputs — only in Notebook 06 evaluation.


In [ ]:
# Imports and path setup
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Add src/ to path so we can import our modules
sys.path.insert(0, os.path.join('..', 'src'))

from data_generation import generate_full_dataset, print_dataset_summary, FLEET

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
print('Imports OK')


## 1. Generate the Full Dataset

This calls `generate_full_dataset()` which:
1. Creates timestamps at 15-min intervals for 30 days (2,880 per server)
2. Generates realistic baseline metrics per server with diurnal patterns + random-walk drift
3. Injects ~5% anomalies (cpu_spike, memory_leak, latency_spike, error_cascade, disk_bottleneck)
4. Saves to `data/raw/infra_logs.csv`


In [ ]:
print('Generating dataset for 15 servers x 30 days x 96 readings/day...\n')

df = generate_full_dataset(
    master_seed=42,
    days=30,
    interval_minutes=15,
    output_path='../data/raw/infra_logs.csv',
    target_anomaly_rate=0.05,
)


## 2. Verification Summary


In [ ]:
print_dataset_summary(df)
print('\nFirst 5 rows:')
df.head()


## 3. Sanity Check — Anomaly Rate Within Target

Confirm the overall anomaly rate is in the 4-6% range.


In [ ]:
rate = df['is_anomaly'].mean() * 100
assert 4.0 <= rate <= 7.0, f'Anomaly rate {rate:.2f}% is outside the 4-6% target!'
print(f'Anomaly rate: {rate:.2f}% -- within target range (4-6%)')


## 4. Sanity Plot — CPU / Memory / Latency (One Server, First 7 Days)


In [ ]:
srv = 'srv-web-01'
srv_df = df[df['server_id'] == srv].copy()
srv_df['timestamp'] = pd.to_datetime(srv_df['timestamp'])

mask_week = srv_df['timestamp'] < srv_df['timestamp'].min() + pd.Timedelta(days=7)
week_df = srv_df[mask_week]

fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True)
fig.suptitle(f'{srv} -- First 7 Days (red = injected anomaly)', fontsize=14, fontweight='bold')

metrics = ['cpu_utilization_pct', 'memory_utilization_pct', 'network_latency_ms']
labels  = ['CPU Utilization (%)', 'Memory Utilization (%)', 'Network Latency (ms)']
colors  = ['steelblue', 'mediumseagreen', 'darkorange']

for ax, metric, label, color in zip(axes, metrics, labels, colors):
    ax.plot(week_df['timestamp'], week_df[metric], color=color, lw=0.8, alpha=0.85)
    anom = week_df[week_df['is_anomaly'] == 1]
    ax.scatter(anom['timestamp'], anom[metric], color='crimson', s=18, zorder=5,
               label='Injected anomaly')
    ax.set_ylabel(label, fontsize=9)
    ax.legend(loc='upper right', fontsize=8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../results/sanity_plot_srv_web_01.png', dpi=120, bbox_inches='tight')
plt.show()
print('> Ground truth labels used here ONLY for visual sanity checking -- never for model training.')


## 5. Data Schema and Descriptive Statistics


In [ ]:
print('Shape:', df.shape)
print('\nColumn dtypes:')
print(df.dtypes)
print('\nNull values per column:')
print(df.isnull().sum())
print('\nDescriptive statistics:')
df.describe().round(2)


## Notebook 01 Complete

**What was generated:**
- `data/raw/infra_logs.csv` -- 43,200 rows, 15 servers, 30 days
- 5 anomaly types injected at ~5% overall rate
- Diurnal patterns, random-walk drift, and cross-metric correlations present

**Next:** Run `02_eda.ipynb` for exploratory analysis.

> Label isolation maintained: `is_anomaly` used only for visual verification.
